# Assignment 5 – PDE for an option on the maximum of two stocks (20 points)

$S_1(t)$, $S_2(t)$ are independent stock processes with equal volatility, $S_1(0) = S_2(0) = 100$. The contract pays $V(T) = \max\{S_1(T), S_2(T)\}$.

a) Fair value formula, b) the PDE, c) its finite difference solution, d) comparison with Monte Carlo.

**Model and parameters.** Black–Scholes model for both stocks, $dS_i = \mu S_i\,dt + \sigma S_i\,dW_i$, $W_1 \perp W_2$.
The task fixes only the spots; the other parameters are taken from the course market data: $S_0 = 100$ is the spot of `TEST_COMPANY`, whose
1-year at-the-money volatility in `mkt/vol_test_company.csv` is $\sigma = 30\%$, so $T = 1$; the risk-free rate is `MarketData.risk_free_rate` $= 5\%$.

In [1]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.sparse.linalg import splu
from scipy.stats import norm

sigma, rate, maturity, spot = 0.30, 0.05, 1.0, 100.0

## a) Fair value

$$ V(t, S_1, S_2) = S_1 N(d_1) + S_2 N(-d_2), \qquad d_1 = \frac{\ln(S_1/S_2) + \tfrac12 \hat\sigma^2 \tau}{\hat\sigma\sqrt{\tau}}, \quad d_2 = d_1 - \hat\sigma\sqrt{\tau}, \quad \hat\sigma = \sqrt{2}\,\sigma, \quad \tau = T - t . $$

For $S_1(0) = S_2(0) = S_0$: $\; d_1 = \sigma\sqrt{T/2} = -d_2$, so
$$ V(0) = 2 S_0\, N\!\left(\sigma\sqrt{T/2}\right). $$

The formula is derived from the PDE at the end of part b). It does not contain the interest rate.

In [2]:
def max_option_analytic(s1, s2, tau, sigma):
    vol = np.sqrt(2) * sigma * np.sqrt(tau)
    d1 = (np.log(s1 / s2) + 0.5 * vol**2) / vol
    return s1 * norm.cdf(d1) + s2 * norm.cdf(-(d1 - vol))


analytic = max_option_analytic(spot, spot, maturity, sigma)
print(f'Analytic fair value: V(0) = 2 * 100 * N(0.3 * sqrt(0.5)) = {analytic:.6f}')

Analytic fair value: V(0) = 2 * 100 * N(0.3 * sqrt(0.5)) = 116.799597


## b) The PDE

As for the Black–Scholes PDE in the BSM lecture, now with two stocks. By Itô's lemma for $V(t, S_1, S_2)$, using that $W_1$ and $W_2$ are independent (no cross term):
$$ dV = \left(V_t + \tfrac12\sigma^2 S_1^2 V_{S_1S_1} + \tfrac12\sigma^2 S_2^2 V_{S_2S_2}\right)dt + V_{S_1}\,dS_1 + V_{S_2}\,dS_2 . $$
The portfolio $\Pi = V - V_{S_1} S_1 - V_{S_2} S_2$ (short the derivative's deltas in both stocks) has no $dW_1$, $dW_2$ terms, so it is riskless and earns the risk-free rate, $d\Pi = r\Pi\,dt$:
$$ \boxed{\; V_t + \tfrac12\sigma^2 S_1^2 V_{S_1S_1} + \tfrac12\sigma^2 S_2^2 V_{S_2S_2} + r S_1 V_{S_1} + r S_2 V_{S_2} - rV = 0, \qquad V(T, S_1, S_2) = \max\{S_1, S_2\} \;} $$

**Boundary conditions** on the domain $[0, S_{max}]^2$:
* $S_1 = 0$: $S_1$ stays at 0, the contract pays $S_2(T)$, which is worth $S_2$ today: $V = S_2$. Similarly $V = S_1$ at $S_2 = 0$.
* $S_1 = S_{max}$, $S_2 = S_{max}$: $V_{S_1S_1} = 0$, $V_{S_2S_2} = 0$. Far from the other stock the contract behaves like the stock itself, whose value is linear in its price.

**Derivation of the formula in a).** The payoff is homogeneous: $\max\{\lambda S_1, \lambda S_2\} = \lambda \max\{S_1, S_2\}$, so try $V(t, S_1, S_2) = S_2\, W(t, x)$ with $x = S_1/S_2$.
Then $V_{S_1} = W_x$, $V_{S_1S_1} = W_{xx}/S_2$, $V_{S_2} = W - xW_x$, $V_{S_2S_2} = x^2 W_{xx}/S_2$, and the PDE becomes
$$ S_2\left(W_t + \tfrac12\sigma^2x^2W_{xx} + \tfrac12\sigma^2x^2W_{xx}\right) + rS_1W_x + rS_2(W - xW_x) - rS_2W = 0 \iff W_t + \tfrac12 \hat\sigma^2 x^2 W_{xx} = 0, \quad \hat\sigma^2 = 2\sigma^2, $$
with $W(T, x) = \max\{x, 1\} = 1 + (x - 1)^+$. This is the Black–Scholes PDE with zero rate and volatility $\hat\sigma$. The constant 1 solves it, and $(x-1)^+$ is the payoff of a call with strike 1,
whose solution is the BSM call formula of the lecture: $x N(d_1) - N(d_2)$ (with $r = 0$, $\hat\sigma$). By linearity $W = 1 + x N(d_1) - N(d_2)$, so
$$ V = S_2 W = S_2 + S_1 N(d_1) - S_2 N(d_2) = S_1 N(d_1) + S_2 N(-d_2). $$

## c) Finite difference solution: Crank–Nicolson on a 2D grid

The scheme of the PDE lecture extended to two dimensions. Uniform grid $S_i = jh$, $j = 0, \dots, N$, $h = S_{max}/N$; unknowns at the $(N-1)^2$ interior nodes.
In each direction central differences, as in the finite difference lecture:
$$ V_{SS} \approx \frac{V_{j+1} - 2V_j + V_{j-1}}{h^2}, \qquad V_S \approx \frac{V_{j+1} - V_{j-1}}{2h} . $$
* At $S = 0$ the boundary value is known and enters the first interior node as a constant source term $b$.
* At $S_{max}$ the condition $V_{SS} = 0$ means $V_N = 2V_{N-1} - V_{N-2}$. Substituted into the last interior node, its second difference vanishes and its first difference becomes the backward difference.

With the 1D operator $A = \mathrm{diag}(\tfrac12\sigma^2 S^2) D_2 + \mathrm{diag}(rS) D_1$, the 2D operator is $L = A \otimes I + I \otimes A - rI$,
and in time to maturity $\tau$: $\; \partial_\tau V = LV + b$. Crank–Nicolson step:
$$ \left(I - \tfrac{\Delta\tau}{2}L\right)V^{m+1} = \left(I + \tfrac{\Delta\tau}{2}L\right)V^m + \Delta\tau\, b . $$
The matrix is the same in every step, so it is LU-factorized once.

In [3]:
def solve_max_option_pde(s_max: float, und_step: float, time_step: float, sigma: float = sigma, rate: float = rate,
                         maturity: float = maturity) -> tuple[np.ndarray, np.ndarray]:
    """Crank-Nicolson solution of the 2D PDE. Returns the interior grid S and V(0, S1, S2) on it (V[i, j]: S1 = S[i], S2 = S[j])."""
    h = und_step
    s = np.linspace(0, s_max, int(round(s_max / h)) + 1)[1:-1]
    n = len(s)

    # 1D difference operators on the interior nodes
    ones = np.ones(n - 1)
    d2 = sp.diags([ones, -2 * np.ones(n), ones], [-1, 0, 1], format='lil') / h**2
    d1 = sp.diags([-ones, ones], [-1, 1], format='lil') / (2 * h)
    # S_max: V_N = 2 V_{N-1} - V_{N-2}  =>  no second difference, backward first difference
    d2[n - 1, :] = 0
    d1[n - 1, n - 2], d1[n - 1, n - 1] = -1 / h, 1 / h
    a = sp.diags(0.5 * sigma**2 * s**2) @ d2.tocsr() + sp.diags(rate * s) @ d1.tocsr()

    # 2D operator, node (i, j) at position i * n + j
    eye = sp.identity(n)
    op = sp.kron(a, eye) + sp.kron(eye, a) - rate * sp.identity(n * n)

    # S = 0 boundary (V = S2 at S1 = 0, V = S1 at S2 = 0) enters the first interior node (S = h)
    # with weight 0.5 sigma^2 S^2 / h^2 - r S / (2h) = 0.5 sigma^2 - 0.5 r
    b = np.zeros((n, n))
    weight = 0.5 * sigma**2 - 0.5 * rate
    b[0, :] += weight * s
    b[:, 0] += weight * s

    v = np.maximum.outer(s, s).ravel()  # payoff at maturity
    n_time = int(round(maturity / time_step))
    lhs = (sp.identity(n * n) - 0.5 * time_step * op).tocsc()
    rhs = (sp.identity(n * n) + 0.5 * time_step * op).tocsr()
    lu = splu(lhs)
    for _ in range(n_time):
        v = lu.solve(rhs @ v + time_step * b.ravel())
    return s, v.reshape(n, n)

Grid parameters: the defaults of `PDEParams` in the PDE lecture, `stock_max_mult = 2` ($S_{max} = 200$), `und_step = 2`, `time_step = 1/1200`.

In [4]:
s, v = solve_max_option_pde(s_max=2 * spot, und_step=2, time_step=1 / 1200)
i = int(np.argmin(np.abs(s - spot)))
pde_value = v[i, i]
print(f'PDE fair value: {pde_value:.6f}, error: {(pde_value - analytic) / analytic * 100:.4f}%')

PDE fair value: 116.779381, error: -0.0173%


## d) Monte Carlo

Under the risk-neutral measure the terminal spots are $S_i(T) = S_0 \exp\left((r - \tfrac12\sigma^2)T + \sigma\sqrt{T} Z_i\right)$ with independent standard normal $Z_1, Z_2$
(the solution of the geometric Brownian motion), so they are simulated directly at $T$, and $V(0) \approx e^{-rT}\frac{1}{M}\sum_k \max\{S_1^{(k)}(T), S_2^{(k)}(T)\}$.

The random numbers are generated as in the course library (`MCMethod.generate_std_norm` with the `MCParams` defaults): seed 1, 10 000 paths, antithetic pairs, standardized.
The antithetic pairs $(Z, -Z)$ are not independent, so the standard error is computed from the 5 000 pair averages, which are.

In [5]:
num_of_paths = 10000
np.random.seed(1)
rnd1 = np.random.standard_normal(size=(num_of_paths // 2, 2))  # one column per stock
rnd = np.concatenate((rnd1, -rnd1), axis=0)                      # antithetic
rnd = (rnd - np.mean(rnd)) / np.std(rnd, ddof=1)                 # standardize

terminal = spot * np.exp((rate - 0.5 * sigma**2) * maturity + sigma * np.sqrt(maturity) * rnd)
discounted_payoff = np.exp(-rate * maturity) * np.maximum(terminal[:, 0], terminal[:, 1])
pair_averages = 0.5 * (discounted_payoff[:num_of_paths // 2] + discounted_payoff[num_of_paths // 2:])
mc_value = discounted_payoff.mean()
mc_std_error = pair_averages.std(ddof=1) / np.sqrt(num_of_paths // 2)

pd.DataFrame({
    'fair value': [analytic, pde_value, mc_value],
    'difference to analytic': [0.0, pde_value - analytic, mc_value - analytic],
    'MC std. error': [np.nan, np.nan, mc_std_error],
}, index=['analytic', 'PDE, Crank–Nicolson', 'Monte Carlo, 10 000 paths'])

,fair value,difference to analytic,MC std. error
analytic,116.799597,0.000000,NaN
"PDE, Crank–Nicolson",116.779381,-0.020216,NaN
"Monte Carlo, 10 000 paths",116.999933,0.200335,0.235364


**Conclusion.** The three methods agree:
* analytic: $V(0) = 2 S_0 N(\sigma\sqrt{T/2}) = 116.7996$;
* PDE: $116.7794$, error $-0.017\%$;
* Monte Carlo: $117.0000$, difference $0.20$, which is $0.85$ standard errors (standard error $0.24$), i.e. within the statistical uncertainty.